# Question 2: Text Preprocessing Pipeline

Cleaning, Tokenization, Stopword Removal, Stemming (Porter & Snowball) and Lemmatization
on noisy health & fitness sentences using **pandas** and **NLTK**.

## Step 0: Import libraries

In [1]:
import re
import nltk
import pandas as pd
from nltk.tokenize import word_tokenize, RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/sayanbhattacharya/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/sayanbhattacharya/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/sayanbhattacharya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/sayanbhattacharya/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/sayanbhattacharya/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

## Step 1: Load the raw sentences into a DataFrame

In [2]:
raw_sentences = [
    "A balanced DIET helps maintain healthy body weight!!",
    "regular workout sessions improve muscle strength... love it :)",
    "drinking 8 glasses of water daily supports digestion!!",
    "skipping meals can SLOW down your metabolism...",
    "stretching before a workout reduces injury risk!! important",
    "strength training builds muscle & improves bone density",
    "getting 7-8 hrs of sleep is important for recovery :)",
    "yoga is a LOW-impact workout, improves flexibility!!",
    "a high-sugar diet is linked to health problems...",
    "interval training burns calories QUICKLY!! great results",
    "nutritionists design diet plans based on individual goals",
    "rest days are an important part of any workout schedule!!"
]

df = pd.DataFrame({"text": raw_sentences})

# save the dataset as dataset.csv
df.to_csv("dataset.csv", index=False)

df

,text
0,A balanced DIET helps maintain healthy body we...
1,regular workout sessions improve muscle streng...
2,drinking 8 glasses of water daily supports dig...
3,skipping meals can SLOW down your metabolism...
4,stretching before a workout reduces injury ris...
5,strength training builds muscle & improves bon...
6,getting 7-8 hrs of sleep is important for reco...
7,"yoga is a LOW-impact workout, improves flexibi..."
8,a high-sugar diet is linked to health problems...
9,interval training burns calories QUICKLY!! gre...


## Step 2: Cleaning function

The function will:
1. make the text lowercase
2. remove punctuation, numbers and special characters (keep only letters and spaces)
3. remove extra spaces

In [3]:
def clean_text(text):
    text = text.lower()                       # 1. lowercase
    text = re.sub(r'[^a-z\s]', ' ', text)     # 2. keep only letters and spaces
    text = re.sub(r'\s+', ' ', text)          # 3. collapse extra spaces
    text = text.strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)

Raw and cleaned text side by side:

In [4]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 100)
df[["text", "clean_text"]]

,text,clean_text
0,A balanced DIET helps maintain healthy body weight!!,a balanced diet helps maintain healthy body weight
1,regular workout sessions improve muscle strength... love it :),regular workout sessions improve muscle strength love it
2,drinking 8 glasses of water daily supports digestion!!,drinking glasses of water daily supports digestion
3,skipping meals can SLOW down your metabolism...,skipping meals can slow down your metabolism
4,stretching before a workout reduces injury risk!! important,stretching before a workout reduces injury risk important
5,strength training builds muscle & improves bone density,strength training builds muscle improves bone density
6,getting 7-8 hrs of sleep is important for recovery :),getting hrs of sleep is important for recovery
7,"yoga is a LOW-impact workout, improves flexibility!!",yoga is a low impact workout improves flexibility
8,a high-sugar diet is linked to health problems...,a high sugar diet is linked to health problems
9,interval training burns calories QUICKLY!! great results,interval training burns calories quickly great results


## Step 3: Tokenization (word_tokenize vs RegexpTokenizer)

In [5]:
# (a) NLTK word_tokenize
df["word_tokens"] = df["clean_text"].apply(word_tokenize)

# (b) RegexpTokenizer that keeps only alphabetic words
regexp_tokenizer = RegexpTokenizer(r'[a-zA-Z]+')
df["regexp_tokens"] = df["clean_text"].apply(regexp_tokenizer.tokenize)

for i in range(3):
    print("Sentence", i + 1, ":", df["clean_text"][i])
    print("word_tokenize   :", df["word_tokens"][i])
    print("RegexpTokenizer :", df["regexp_tokens"][i])
    print()

Sentence 1 : a balanced diet helps maintain healthy body weight
word_tokenize   : ['a', 'balanced', 'diet', 'helps', 'maintain', 'healthy', 'body', 'weight']
RegexpTokenizer : ['a', 'balanced', 'diet', 'helps', 'maintain', 'healthy', 'body', 'weight']

Sentence 2 : regular workout sessions improve muscle strength love it
word_tokenize   : ['regular', 'workout', 'sessions', 'improve', 'muscle', 'strength', 'love', 'it']
RegexpTokenizer : ['regular', 'workout', 'sessions', 'improve', 'muscle', 'strength', 'love', 'it']

Sentence 3 : drinking glasses of water daily supports digestion
word_tokenize   : ['drinking', 'glasses', 'of', 'water', 'daily', 'supports', 'digestion']
RegexpTokenizer : ['drinking', 'glasses', 'of', 'water', 'daily', 'supports', 'digestion']



Because we already cleaned the text (no punctuation or numbers left), both tokenizers give the **same** output here.
To see the real difference, let us also try both tokenizers on the **raw** text of the first 3 sentences:

In [6]:
for i in range(3):
    print("Raw sentence", i + 1, ":", df["text"][i])
    print("word_tokenize   :", word_tokenize(df["text"][i]))
    print("RegexpTokenizer :", regexp_tokenizer.tokenize(df["text"][i]))
    print()

Raw sentence 1 : A balanced DIET helps maintain healthy body weight!!
word_tokenize   : ['A', 'balanced', 'DIET', 'helps', 'maintain', 'healthy', 'body', 'weight', '!', '!']
RegexpTokenizer : ['A', 'balanced', 'DIET', 'helps', 'maintain', 'healthy', 'body', 'weight']

Raw sentence 2 : regular workout sessions improve muscle strength... love it :)
word_tokenize   : ['regular', 'workout', 'sessions', 'improve', 'muscle', 'strength', '...', 'love', 'it', ':', ')']
RegexpTokenizer : ['regular', 'workout', 'sessions', 'improve', 'muscle', 'strength', 'love', 'it']

Raw sentence 3 : drinking 8 glasses of water daily supports digestion!!
word_tokenize   : ['drinking', '8', 'glasses', 'of', 'water', 'daily', 'supports', 'digestion', '!', '!']
RegexpTokenizer : ['drinking', 'glasses', 'of', 'water', 'daily', 'supports', 'digestion']



**Difference:** On raw text, `word_tokenize` keeps punctuation (`!!`, `...`, `:`, `)`) and numbers (`8`) as separate
tokens, while `RegexpTokenizer` with the pattern `[a-zA-Z]+` keeps only the alphabetic words. On cleaned text both are the same.

## Step 4: Stopword removal

In [7]:
stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    result = []
    for word in tokens:
        if word not in stop_words:
            result.append(word)
    return result

df["no_stopwords"] = df["word_tokens"].apply(remove_stopwords)

df["count_before"] = df["word_tokens"].apply(len)
df["count_after"] = df["no_stopwords"].apply(len)

df[["clean_text", "count_before", "count_after"]]

,clean_text,count_before,count_after
0,a balanced diet helps maintain healthy body weight,8,7
1,regular workout sessions improve muscle strength love it,8,7
2,drinking glasses of water daily supports digestion,7,6
3,skipping meals can slow down your metabolism,7,4
4,stretching before a workout reduces injury risk important,8,6
5,strength training builds muscle improves bone density,7,7
6,getting hrs of sleep is important for recovery,8,5
7,yoga is a low impact workout improves flexibility,8,6
8,a high sugar diet is linked to health problems,9,6
9,interval training burns calories quickly great results,7,7


In [8]:
for i in range(len(df)):
    print(i + 1, df["no_stopwords"][i])

1 ['balanced', 'diet', 'helps', 'maintain', 'healthy', 'body', 'weight']
2 ['regular', 'workout', 'sessions', 'improve', 'muscle', 'strength', 'love']
3 ['drinking', 'glasses', 'water', 'daily', 'supports', 'digestion']
4 ['skipping', 'meals', 'slow', 'metabolism']
5 ['stretching', 'workout', 'reduces', 'injury', 'risk', 'important']
6 ['strength', 'training', 'builds', 'muscle', 'improves', 'bone', 'density']
7 ['getting', 'hrs', 'sleep', 'important', 'recovery']
8 ['yoga', 'low', 'impact', 'workout', 'improves', 'flexibility']
9 ['high', 'sugar', 'diet', 'linked', 'health', 'problems']
10 ['interval', 'training', 'burns', 'calories', 'quickly', 'great', 'results']
11 ['nutritionists', 'design', 'diet', 'plans', 'based', 'individual', 'goals']
12 ['rest', 'days', 'important', 'part', 'workout', 'schedule']


## Step 5: Porter Stemmer

In [9]:
porter = PorterStemmer()

def porter_stem(tokens):
    result = []
    for word in tokens:
        result.append(porter.stem(word))
    return result

df["porter"] = df["no_stopwords"].apply(porter_stem)

for i in range(len(df)):
    print(i + 1, df["porter"][i])

1 ['balanc', 'diet', 'help', 'maintain', 'healthi', 'bodi', 'weight']
2 ['regular', 'workout', 'session', 'improv', 'muscl', 'strength', 'love']
3 ['drink', 'glass', 'water', 'daili', 'support', 'digest']
4 ['skip', 'meal', 'slow', 'metabol']
5 ['stretch', 'workout', 'reduc', 'injuri', 'risk', 'import']
6 ['strength', 'train', 'build', 'muscl', 'improv', 'bone', 'densiti']
7 ['get', 'hr', 'sleep', 'import', 'recoveri']
8 ['yoga', 'low', 'impact', 'workout', 'improv', 'flexibl']
9 ['high', 'sugar', 'diet', 'link', 'health', 'problem']
10 ['interv', 'train', 'burn', 'calori', 'quickli', 'great', 'result']
11 ['nutritionist', 'design', 'diet', 'plan', 'base', 'individu', 'goal']
12 ['rest', 'day', 'import', 'part', 'workout', 'schedul']


## Step 6: Snowball Stemmer (English)

In [10]:
snowball = SnowballStemmer('english')

def snowball_stem(tokens):
    result = []
    for word in tokens:
        result.append(snowball.stem(word))
    return result

df["snowball"] = df["no_stopwords"].apply(snowball_stem)

for i in range(len(df)):
    print(i + 1, df["snowball"][i])

1 ['balanc', 'diet', 'help', 'maintain', 'healthi', 'bodi', 'weight']
2 ['regular', 'workout', 'session', 'improv', 'muscl', 'strength', 'love']
3 ['drink', 'glass', 'water', 'daili', 'support', 'digest']
4 ['skip', 'meal', 'slow', 'metabol']
5 ['stretch', 'workout', 'reduc', 'injuri', 'risk', 'import']
6 ['strength', 'train', 'build', 'muscl', 'improv', 'bone', 'densiti']
7 ['get', 'hrs', 'sleep', 'import', 'recoveri']
8 ['yoga', 'low', 'impact', 'workout', 'improv', 'flexibl']
9 ['high', 'sugar', 'diet', 'link', 'health', 'problem']
10 ['interv', 'train', 'burn', 'calori', 'quick', 'great', 'result']
11 ['nutritionist', 'design', 'diet', 'plan', 'base', 'individu', 'goal']
12 ['rest', 'day', 'import', 'part', 'workout', 'schedul']


## Step 7: WordNet Lemmatizer

By default the lemmatizer treats every word as a noun. Verbs like *improves* or *builds* would not change,
so we first try the word as a verb, and if nothing changes we try it as a noun.

In [11]:
lemmatizer = WordNetLemmatizer()

def lemmatize_word(word):
    lemma = lemmatizer.lemmatize(word, pos='v')    # try as a verb
    if lemma == word:
        lemma = lemmatizer.lemmatize(word, pos='n')  # try as a noun
    return lemma

def lemmatize_tokens(tokens):
    result = []
    for word in tokens:
        result.append(lemmatize_word(word))
    return result

df["lemma"] = df["no_stopwords"].apply(lemmatize_tokens)

for i in range(len(df)):
    print(i + 1, df["lemma"][i])

1 ['balance', 'diet', 'help', 'maintain', 'healthy', 'body', 'weight']
2 ['regular', 'workout', 'session', 'improve', 'muscle', 'strength', 'love']
3 ['drink', 'glass', 'water', 'daily', 'support', 'digestion']
4 ['skip', 'meal', 'slow', 'metabolism']
5 ['stretch', 'workout', 'reduce', 'injury', 'risk', 'important']
6 ['strength', 'train', 'build', 'muscle', 'improve', 'bone', 'density']
7 ['get', 'hr', 'sleep', 'important', 'recovery']
8 ['yoga', 'low', 'impact', 'workout', 'improve', 'flexibility']
9 ['high', 'sugar', 'diet', 'link', 'health', 'problem']
10 ['interval', 'train', 'burn', 'calorie', 'quickly', 'great', 'result']
11 ['nutritionist', 'design', 'diet', 'plan', 'base', 'individual', 'goal']
12 ['rest', 'day', 'important', 'part', 'workout', 'schedule']


All outputs side by side:

In [12]:
df[["no_stopwords", "porter", "snowball", "lemma"]]

,no_stopwords,porter,snowball,lemma
0,"[balanced, diet, helps, maintain, healthy, body, weight]","[balanc, diet, help, maintain, healthi, bodi, weight]","[balanc, diet, help, maintain, healthi, bodi, weight]","[balance, diet, help, maintain, healthy, body, weight]"
1,"[regular, workout, sessions, improve, muscle, strength, love]","[regular, workout, session, improv, muscl, strength, love]","[regular, workout, session, improv, muscl, strength, love]","[regular, workout, session, improve, muscle, strength, love]"
2,"[drinking, glasses, water, daily, supports, digestion]","[drink, glass, water, daili, support, digest]","[drink, glass, water, daili, support, digest]","[drink, glass, water, daily, support, digestion]"
3,"[skipping, meals, slow, metabolism]","[skip, meal, slow, metabol]","[skip, meal, slow, metabol]","[skip, meal, slow, metabolism]"
4,"[stretching, workout, reduces, injury, risk, important]","[stretch, workout, reduc, injuri, risk, import]","[stretch, workout, reduc, injuri, risk, import]","[stretch, workout, reduce, injury, risk, important]"
5,"[strength, training, builds, muscle, improves, bone, density]","[strength, train, build, muscl, improv, bone, densiti]","[strength, train, build, muscl, improv, bone, densiti]","[strength, train, build, muscle, improve, bone, density]"
6,"[getting, hrs, sleep, important, recovery]","[get, hr, sleep, import, recoveri]","[get, hrs, sleep, import, recoveri]","[get, hr, sleep, important, recovery]"
7,"[yoga, low, impact, workout, improves, flexibility]","[yoga, low, impact, workout, improv, flexibl]","[yoga, low, impact, workout, improv, flexibl]","[yoga, low, impact, workout, improve, flexibility]"
8,"[high, sugar, diet, linked, health, problems]","[high, sugar, diet, link, health, problem]","[high, sugar, diet, link, health, problem]","[high, sugar, diet, link, health, problem]"
9,"[interval, training, burns, calories, quickly, great, results]","[interv, train, burn, calori, quickli, great, result]","[interv, train, burn, calori, quick, great, result]","[interval, train, burn, calorie, quickly, great, result]"


## Step 8: Comparison table (Original | Porter | Snowball | Lemma)

We take all distinct words (after stopword removal) and compare the three methods.
The last column shows **DIFFERENT** where Porter and Snowball disagree.

In [13]:
all_words = []
for tokens in df["no_stopwords"]:
    for word in tokens:
        if word not in all_words:
            all_words.append(word)

rows = []
for word in all_words:
    p = porter.stem(word)
    s = snowball.stem(word)
    l = lemmatize_word(word)
    if p == s:
        check = "same"
    else:
        check = "DIFFERENT"
    rows.append([word, p, s, l, check])

compare = pd.DataFrame(rows, columns=["Original Word", "Porter Stem", "Snowball Stem", "Lemma", "Porter vs Snowball"])
compare

,Original Word,Porter Stem,Snowball Stem,Lemma,Porter vs Snowball
0,balanced,balanc,balanc,balance,same
1,diet,diet,diet,diet,same
2,helps,help,help,help,same
3,maintain,maintain,maintain,maintain,same
4,healthy,healthi,healthi,healthy,same
5,body,bodi,bodi,body,same
6,weight,weight,weight,weight,same
7,regular,regular,regular,regular,same
8,workout,workout,workout,workout,same
9,sessions,session,session,session,same


Words where Porter and Snowball disagree:

In [14]:
compare[compare["Porter vs Snowball"] == "DIFFERENT"]

,Original Word,Porter Stem,Snowball Stem,Lemma,Porter vs Snowball
35,hrs,hr,hrs,hr,DIFFERENT
50,quickly,quickli,quick,quickly,DIFFERENT


## Step 9: Vocabulary size at each stage

In [15]:
def vocab_size(column):
    words = set()
    for tokens in df[column]:
        for word in tokens:
            words.add(word)
    return len(words)

print("Raw tokens (after cleaning + tokenizing):", vocab_size("word_tokens"))
print("After stopword removal                  :", vocab_size("no_stopwords"))
print("After Porter stemming                   :", vocab_size("porter"))
print("After Snowball stemming                 :", vocab_size("snowball"))
print("After lemmatization                     :", vocab_size("lemma"))

Raw tokens (after cleaning + tokenizing): 77
After stopword removal                  : 63
After Porter stemming                   : 62
After Snowball stemming                 : 62
After lemmatization                     : 62


## Step 10: Interpretation

Stemming and lemmatization both reduced the vocabulary a little (63 → 62 words), because our dataset is small and
most words appear in only one form. The big difference is in the quality of the output: the stemmers produced
broken words like *balanc*, *healthi*, *bodi*, *daili*, *quickli* and *schedul*, while the lemmatizer gave real words
like *balance*, *healthy*, *body*, *daily* and *schedule*. Porter and Snowball gave almost the same results, Snowball
was only a little smarter (*quickly → quick*). For a search engine I would prefer **stemming** because it is fast and
it matches many forms of a word together (*improve*, *improves*, *improving* → *improv*). For a classification task or
anything where people read the output, I would prefer **lemmatization** because the words stay meaningful and
easy to understand.

## Questions for Interpretation

**1. Why can stemming produce word forms that are not valid dictionary words, while lemmatization always returns a valid word?**

A stemmer just follows fixed rules that cut off endings like *-ing*, *-es*, *-ly*, *-ity*. It does not check a
dictionary, so the result can be a word that does not exist, for example *daily → daili*, *density → densiti*,
*balanced → balanc*. A lemmatizer looks the word up in the WordNet dictionary and returns its base form (lemma),
so the output is always a real word, for example *improves → improve*, *calories → calorie*.

**2. What is the practical difference between using RegexpTokenizer and NLTK's default word_tokenize, and when would you prefer one over the other?**

`word_tokenize` is a smart tokenizer: it splits punctuation into separate tokens and handles things like
contractions (*don't → do, n't*). `RegexpTokenizer` simply keeps whatever matches the pattern we give it, so with
`[a-zA-Z]+` it returns only words and drops numbers and symbols. I would use `word_tokenize` when punctuation is
useful (for example sentence structure or sentiment like "!!"), and `RegexpTokenizer` when I only want clean words
quickly and want full control over what counts as a token.

**3. Find one word from your dataset where the Porter and Snowball stemmers give different results. Explain why the two algorithms might disagree.**

In our dataset two words give different results:
- **quickly**: Porter → *quickli*, Snowball → *quick*
- **hrs**: Porter → *hr*, Snowball → *hrs*

Snowball (also called "Porter2") is an improved version of the Porter stemmer. It has extra and updated rules,
for example a rule that removes the *-ly* ending (so *quickly* becomes *quick*), while Porter only changes the final
*y* to *i*. Snowball also does not remove the final *s* from very short words like *hrs*, but Porter does.
Because the two algorithms use slightly different rule lists, they sometimes give different stems.